# 03 · UNSEEN analysis: how bad can pre-monsoon fire weather get in India — today and at 1.5 / 2 / 3 °C?
Inputs (from notebooks 01–02, all on Drive): CMIP6 large-ensemble FWI indicators, GWL windows, ERA5 indicators & regions.

**Steps**
1. Regional series (area-weighted over the 2.5° cells overlapping each state polygon)
2. Model evaluation vs ERA5 (1985–2014): distributions + bias maps
3. UNSEEN fidelity test (Thompson et al. 2017; Kelder et al. 2022): bootstrap mean / sd / skewness / kurtosis
4. Bias correction (mean–variance scaling to ERA5, per model, region, indicator)
5. Pools of "plausible years" at each warming level (20-yr window × all members), today ≈ 1.3 °C
6. Exceedance (return-period) curves; worst plausible events; P(beating the observed record) per year and per decade
7. Persistence: fwisa (peak 90-day mean), fwixd (extreme days), fwils (season length)
8. Maps of record-breaking probability (GeoTIFF for QGIS)
9. Why UNSEEN: GEV fit to the short observed record vs the large-ensemble estimate

Pure analysis — no downloads. Runtime ≈ 2–5 min. Results → `MyDrive/FWI_India/results`, `figs`, `maps`.

In [ ]:
# ================= CONFIG =================
import os
BASE = os.environ.get('FWI_BASE', '/content/drive/MyDrive/FWI_India')
HUM = 'hursmin'
OBS_VERSION = 'B'                 # 'B' = ERA5 FWI of 2.5°-mean inputs (model-like, recommended) | 'Aagg'
REF = (1985, 2014)                # evaluation / bias-correction period (calendar years; historical runs)
OBS_START = 1979                  # ERA5 years used as 'observations' (1979+ = satellite era; pre-1979 ERA5 is less reliable)
GWL_ORDER = ['today', '1.5', '2.0', '3.0']
GWL_LABEL = {'today': 'today (≈1.3 °C)', '1.5': '+1.5 °C', '2.0': '+2 °C', '3.0': '+3 °C'}
MODELS = ['CanESM5', 'MIROC6', 'ACCESS-ESM1-5', 'MPI-ESM1-2-LR']
N_BOOT = 10000
SEED = 42

In [ ]:
# [COLAB-ONLY]
from google.colab import drive
drive.mount('/content/drive')
!pip -q install rioxarray cftime netCDF4 geopandas

In [ ]:
import json, warnings
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')
def polygons_only(g):
    """Earth Engine unions can come back as GeometryCollections (polygons + stray lines) → keep polygons only."""
    from shapely.geometry import MultiPolygon
    from shapely.ops import unary_union
    if g is None or g.is_empty or g.geom_type in ('Polygon', 'MultiPolygon'):
        return g
    u = unary_union([p for p in getattr(g, 'geoms', []) if p.geom_type in ('Polygon', 'MultiPolygon')])
    return u if u.geom_type == 'MultiPolygon' else MultiPolygon([u])
RES, FIG, MAPS = f'{BASE}/results', f'{BASE}/figs', f'{BASE}/maps'
for d in (RES, FIG, MAPS):
    os.makedirs(d, exist_ok=True)
rng = np.random.default_rng(SEED)
COL = {'CanESM5': '#1b9e77', 'MIROC6': '#d95f02', 'ACCESS-ESM1-5': '#7570b3', 'MPI-ESM1-2-LR': '#e7298a'}
GCOL = {'today': '#4d4d4d', '1.5': '#fdae61', '2.0': '#f46d43', '3.0': '#a50026'}

## 1. Load data and build regional series

In [ ]:
W = xr.open_dataset(f'{BASE}/data/region_weights_g025.nc')
REGIONS = [v[2:].replace('_', ' ') for v in W.data_vars]
obs_g = xr.open_dataset(f'{BASE}/data/era5/era5_indicators_g025.nc').sel(year=slice(OBS_START, None))
obs_reg = pd.read_csv(f'{BASE}/data/era5/era5_region_annual.csv', header=[0, 1, 2], index_col=0).loc[OBS_START:]
win = pd.read_csv(f'{BASE}/data/cmip6/gwl_windows_ssp585.csv')
win = win[win.in_fwi_data & win.gwl.isin(GWL_ORDER)]

LE = {}
for ind in ['fwixx', 'fwisa', 'fwixd', 'fwils']:
    f = f'{BASE}/data/cmip6/LE_{ind}_{HUM}_india.nc'
    if not os.path.exists(f):
        print(f'⚠ {ind}: file not found ({os.path.basename(f)}) → skipped. Finish notebook 01 for this indicator to include it.')
        continue
    da = xr.open_dataarray(f)
    da = da.where(da.has_hist & da.has_ssp585, drop=True)
    LE[ind] = da.sel(lat=W.lat, lon=W.lon, method='nearest', tolerance=0.01).assign_coords(lat=W.lat, lon=W.lon)
assert 'fwixx' in LE, 'LE_fwixx file is required (run notebook 01)'
IND = list(LE)                      # indicators available for the models
print('model indicators available:', IND)
runs = LE['fwixx'].run.values
print('runs per model:', pd.Series(LE['fwixx'].esm.values).value_counts().to_dict())

def wmean(da, w):
    w = xr.DataArray(w, dims=('lat', 'lon'), coords=dict(lat=da.lat, lon=da.lon))
    return (da * w).sum(('lat', 'lon')) / w.where(da.notnull()).sum(('lat', 'lon'))

MOD = {(reg, ind): wmean(LE[ind], W[f'w_{reg.replace(" ", "_")}'].values).to_pandas()   # DataFrame run × year
       for reg in REGIONS for ind in LE}
OBS = {(reg, ind): obs_reg[(reg, ind, OBS_VERSION)].dropna() for reg in REGIONS for ind in ['fwixx', 'fwisa', 'fwixd', 'fwils']}
esm_of = pd.Series(LE['fwixx'].esm.values, index=runs)
for reg in REGIONS:
    s = OBS[(reg, 'fwixx')]
    print(f'{reg}: ERA5 {int(s.index.min())}–{int(s.index.max())}, record fwixx = {s.max():.1f} ({int(s.idxmax())})')

## 2. Model evaluation vs ERA5 (1985–2014)

In [ ]:
def ref_values(df, runs_sel):
    return df.loc[runs_sel, REF[0]:REF[1]].values.ravel()

fig, axs = plt.subplots(len(REGIONS), 2, figsize=(12, 4 * len(REGIONS)))
evalrows = []
for i, reg in enumerate(REGIONS):
    for j, ind in enumerate([i for i in ['fwixx', 'fwisa'] if i in IND]):
        ax = axs[i, j]
        o = OBS[(reg, ind)].loc[REF[0]:REF[1]].values
        data, labels = [o], ['ERA5']
        for m in MODELS:
            v = ref_values(MOD[(reg, ind)], runs[esm_of == m]); data.append(v); labels.append(m)
            evalrows.append(dict(region=reg, indicator=ind, model=m, obs_mean=o.mean(), mod_mean=v.mean(),
                                 bias=v.mean() - o.mean(), obs_sd=o.std(ddof=1), mod_sd=v.std(ddof=1),
                                 sd_ratio=v.std(ddof=1) / o.std(ddof=1)))
        bp = ax.boxplot(data, patch_artist=True, whis=(1, 99))
        ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
        for patch, c in zip(bp['boxes'], ['k'] + [COL[m] for m in MODELS]):
            patch.set_facecolor(c); patch.set_alpha(.5)
        ax.set_title(f'{reg} · {ind} · {REF[0]}–{REF[1]} (raw models)'); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(f'{FIG}/03_eval_boxplots.png', dpi=150); plt.show()
evaluation = pd.DataFrame(evalrows).round(2); evaluation.to_csv(f'{RES}/03_evaluation_ref.csv', index=False)
evaluation

In [ ]:
regions_gdf = None
try:
    import geopandas as gpd
    regions_gdf = gpd.read_file(f'{MAPS}/regions.geojson')
    regions_gdf['geometry'] = regions_gdf.geometry.apply(polygons_only)
except Exception as e:
    print('regions not plotted:', e)
o_clim = obs_g[f'fwixx_{OBS_VERSION}'].sel(year=slice(*REF)).mean('year')
fig, axs = plt.subplots(1, len(MODELS) + 1, figsize=(4 * (len(MODELS) + 1), 3.6))
o_clim.plot(ax=axs[0], cmap='YlOrRd', vmin=0); axs[0].set_title('ERA5 mean fwixx')
for ax, m in zip(axs[1:], MODELS):
    mc = LE['fwixx'].where(LE['fwixx'].esm == m, drop=True).sel(year=slice(*REF)).mean(('run', 'year'))
    (mc - o_clim).plot(ax=ax, cmap='RdBu_r', vmin=-30, vmax=30); ax.set_title(f'{m} − ERA5')
for ax in axs:
    if regions_gdf is not None:
        regions_gdf.boundary.plot(ax=ax, color='k', lw=.7, aspect=None)
plt.tight_layout(); plt.savefig(f'{FIG}/03_eval_bias_maps.png', dpi=150); plt.show()

## 3. UNSEEN fidelity test
Draw 10 000 samples of 30 member-years from each model's 1985–2014 pool; is the ERA5 value of each statistic inside the
95 % range? Done before and after bias correction (after mean–variance scaling the mean and sd pass by construction, so
**skewness and kurtosis** — the shape of the tail — are the real test).

In [ ]:
def boot_stats(pool, n, nboot=N_BOOT):
    idx = rng.integers(0, pool.size, size=(nboot, n))
    s = pool[idx]
    return dict(mean=s.mean(1), sd=s.std(1, ddof=1), skew=stats.skew(s, axis=1), kurt=stats.kurtosis(s, axis=1))

def obs_stats(o):
    return dict(mean=o.mean(), sd=o.std(ddof=1), skew=stats.skew(o), kurt=stats.kurtosis(o))

def bc_params(reg, ind, m):
    o = OBS[(reg, ind)].loc[REF[0]:REF[1]].values
    v = ref_values(MOD[(reg, ind)], runs[esm_of == m])
    return dict(mu_o=o.mean(), sd_o=o.std(ddof=1), mu_m=v.mean(), sd_m=v.std(ddof=1))

def bias_correct(x, p):
    return (x - p['mu_m']) * (p['sd_o'] / p['sd_m']) + p['mu_o']

fid = []
for reg in REGIONS:
    for ind in [i for i in ['fwixx', 'fwisa'] if i in IND]:
        o = OBS[(reg, ind)].loc[REF[0]:REF[1]].values; so = obs_stats(o)
        for m in MODELS:
            raw = ref_values(MOD[(reg, ind)], runs[esm_of == m])
            for label, pool in [('raw', raw), ('bias-corrected', bias_correct(raw, bc_params(reg, ind, m)))]:
                bs = boot_stats(pool, o.size)
                for k in so:
                    lo, hi = np.percentile(bs[k], [2.5, 97.5])
                    fid.append(dict(region=reg, indicator=ind, model=m, version=label, stat=k, obs=so[k],
                                    lo=lo, hi=hi, passed=lo <= so[k] <= hi))
fid = pd.DataFrame(fid); fid.to_csv(f'{RES}/03_fidelity.csv', index=False)
print(fid[fid.version == 'bias-corrected'].pivot_table(index=['region', 'indicator', 'model'], columns='stat',
      values='passed', aggfunc='first'))

In [ ]:
reg0 = REGIONS[0]
fig, axs = plt.subplots(len(MODELS), 4, figsize=(15, 2.6 * len(MODELS)))
o = OBS[(reg0, 'fwixx')].loc[REF[0]:REF[1]].values; so = obs_stats(o)
for i, m in enumerate(MODELS):
    pool = bias_correct(ref_values(MOD[(reg0, 'fwixx')], runs[esm_of == m]), bc_params(reg0, 'fwixx', m))
    bs = boot_stats(pool, o.size)
    for j, k in enumerate(so):
        ax = axs[i, j]; ax.hist(bs[k], 50, color=COL[m], alpha=.6)
        ax.axvline(so[k], color='k', lw=2); ax.set_title(f'{m} · {k}', fontsize=9)
plt.suptitle(f'Fidelity test, {reg0} fwixx (bias-corrected); black = ERA5'); plt.tight_layout()
plt.savefig(f'{FIG}/03_fidelity_{reg0.replace(" ", "_")}.png', dpi=150); plt.show()

## 4–5. Bias-corrected pools of plausible years at each warming level

In [ ]:
def pool_values(reg, ind, m, gwl, correct=True):
    df = MOD[(reg, ind)]
    w = win[(win.esm == m) & (win.gwl == gwl)]
    vals = [df.loc[r, s:e].values for r, s, e in zip(w.run, w.start, w.end) if r in df.index]
    v = np.concatenate(vals) if vals else np.array([])
    v = v[np.isfinite(v)]
    return bias_correct(v, bc_params(reg, ind, m)) if correct else v

POOLS = {(reg, ind, m, g): pool_values(reg, ind, m, g, correct=ind in ('fwixx', 'fwisa'))
         for reg in REGIONS for ind in IND for m in MODELS for g in GWL_ORDER}
sizes = pd.Series({(m, g): POOLS[(REGIONS[0], 'fwixx', m, g)].size for m in MODELS for g in GWL_ORDER}).unstack()
print('plausible years per pool:\n', sizes[GWL_ORDER])

## 6. Exceedance curves, worst plausible events and record-breaking probabilities

In [ ]:
def exceed_curve(v):
    v = np.sort(v)[::-1]; n = v.size
    p = np.arange(1, n + 1) / (n + 1)               # Weibull plotting position (exceedance probability)
    return 1 / p, v

def wilson(k, n, z=1.645):                           # 90 % CI for a proportion
    if n == 0:
        return np.nan, np.nan
    ph = k / n; d = 1 + z ** 2 / n
    c = (ph + z ** 2 / (2 * n)) / d; h = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2)) / d
    return max(c - h, 0), c + h

rows = []
for reg in REGIONS:
    for ind in [i for i in ['fwixx', 'fwisa'] if i in IND]:
        o = OBS[(reg, ind)]; record, ryear = o.max(), int(o.idxmax())
        for g in GWL_ORDER:
            probs = []
            for m in MODELS:
                v = POOLS[(reg, ind, m, g)]; k = int((v > record).sum()); p = k / v.size
                lo, hi = wilson(k, v.size); probs.append(p)
                rows.append(dict(region=reg, indicator=ind, gwl=g, model=m, n_years=v.size, record=record,
                                 record_year=ryear, n_exceed=k, p_annual=p, p_lo90=lo, p_hi90=hi,
                                 return_period_yr=1 / p if p > 0 else np.inf, p_in_10yr=1 - (1 - p) ** 10,
                                 q99=np.percentile(v, 99), q999=np.percentile(v, 99.9), worst=v.max(),
                                 worst_minus_record=v.max() - record, worst_pct_above_record=100 * (v.max() / record - 1)))
            v_all = np.concatenate([POOLS[(reg, ind, m, g)] for m in MODELS])
            pm = float(np.mean(probs))
            rows.append(dict(region=reg, indicator=ind, gwl=g, model='MULTI-MODEL (equal weight)', n_years=v_all.size,
                             record=record, record_year=ryear, n_exceed=int((v_all > record).sum()), p_annual=pm,
                             p_lo90=min(probs), p_hi90=max(probs), return_period_yr=1 / pm if pm > 0 else np.inf,
                             p_in_10yr=1 - (1 - pm) ** 10, q99=np.percentile(v_all, 99), q999=np.percentile(v_all, 99.9),
                             worst=v_all.max(), worst_minus_record=v_all.max() - record,
                             worst_pct_above_record=100 * (v_all.max() / record - 1)))
results = pd.DataFrame(rows)
results.to_csv(f'{RES}/03_unseen_record_probabilities.csv', index=False)
mm = results[results.model.str.startswith('MULTI')]
print('Multi-model: chance per year of beating the ERA5 record (range across models in brackets)')
print(mm.assign(p=lambda d: (100 * d.p_annual).round(1).astype(str) + '% [' + (100 * d.p_lo90).round(1).astype(str)
               + '–' + (100 * d.p_hi90).round(1).astype(str) + ']').pivot_table(index=['region', 'indicator'], columns='gwl',
               values='p', aggfunc='first')[GWL_ORDER])

In [ ]:
for ind in [i for i in ['fwixx', 'fwisa'] if i in IND]:
    fig, axs = plt.subplots(1, len(REGIONS), figsize=(7 * len(REGIONS), 5), sharey=False)
    axs = np.atleast_1d(axs)
    for ax, reg in zip(axs, REGIONS):
        for g in GWL_ORDER:
            for m in MODELS:
                rp, v = exceed_curve(POOLS[(reg, ind, m, g)]); ax.plot(rp, v, color=GCOL[g], lw=.6, alpha=.45)
            rp, v = exceed_curve(np.concatenate([POOLS[(reg, ind, m, g)] for m in MODELS]))
            ax.plot(rp, v, color=GCOL[g], lw=2.5, label=f'{GWL_LABEL[g]} (n={v.size})')
        o = OBS[(reg, ind)]; rp, v = exceed_curve(o.values)
        ax.scatter(rp, v, color='k', s=14, zorder=5, label=f'ERA5 {int(o.index.min())}–{int(o.index.max())}')
        ax.axhline(o.max(), color='k', ls='--', lw=1); ax.annotate(f'record {int(o.idxmax())}', (1.05, o.max()), va='bottom')
        ax.set_xscale('log'); ax.set_xlabel('return period (years)'); ax.set_ylabel(f'{ind} (bias-corrected)')
        ax.set_title(f'{reg}: {ind}'); ax.grid(alpha=.3, which='both'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG}/03_return_periods_{ind}.png', dpi=150); plt.show()

In [ ]:
fig, axs = plt.subplots(1, len(REGIONS), figsize=(6 * len(REGIONS), 4))
axs = np.atleast_1d(axs)
for ax, reg in zip(axs, REGIONS):
    sub = results[(results.region == reg) & (results.indicator == 'fwixx')]
    for m in MODELS + ['MULTI-MODEL (equal weight)']:
        s = sub[sub.model == m].set_index('gwl').loc[GWL_ORDER]
        kw = dict(color='k', lw=3, marker='o') if m.startswith('MULTI') else dict(color=COL[m], lw=1, marker='.')
        ax.plot(range(len(GWL_ORDER)), 100 * s.p_annual, label=m.replace(' (equal weight)', ''), **kw)
    ax.set_xticks(range(len(GWL_ORDER))); ax.set_xticklabels([GWL_LABEL[g] for g in GWL_ORDER])
    ax.set_ylabel('chance per year of beating the ERA5 record (%)'); ax.set_title(reg); ax.grid(alpha=.3)
axs[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{FIG}/03_record_probability_vs_gwl.png', dpi=150); plt.show()

## 7. Persistence: extreme-fire-weather days (fwixd) and season length (fwils)
These two use model-specific thresholds (95th percentile / mid-range of each member's 1850–1899 FWI), so they are analysed
as **changes within each model** (no bias correction). "Long-season risk" = chance that a year exceeds the 1-in-100-year
value of the model's own 1985–2014 climate.

In [ ]:
PERS = [i for i in ['fwixd', 'fwils'] if i in IND]
if len(PERS) < 2:
    print('Persistence section skipped: needs both fwixd and fwils from notebook 01 (found:', PERS, ')')
else:
    prow = []
    fig, axs = plt.subplots(len(REGIONS), 2, figsize=(13, 4 * len(REGIONS)))
    for i, reg in enumerate(REGIONS):
        for j, ind in enumerate(['fwixd', 'fwils']):
            ax = axs[i, j]; pos = 0; ticks = []
            for m in MODELS:
                ref = ref_values(MOD[(reg, ind)], runs[esm_of == m]); q99 = np.percentile(ref, 99)
                for g in GWL_ORDER:
                    v = POOLS[(reg, ind, m, g)]
                    ax.boxplot(v, positions=[pos], widths=.7, patch_artist=True, showfliers=False,
                               boxprops=dict(facecolor=GCOL[g], alpha=.7))
                    prow.append(dict(region=reg, indicator=ind, model=m, gwl=g, median=np.median(v), q90=np.percentile(v, 90),
                                     ref_median=np.median(ref), p_exceed_ref_1in100=(v > q99).mean(),
                                     likelihood_ratio=(v > q99).mean() / 0.01))
                    pos += 1
                ticks.append(pos - 2.5); pos += 1
            ax.set_xticks(ticks); ax.set_xticklabels(MODELS, fontsize=8)
            ax.set_title(f'{reg} · {ind} ({"days ≥ local 95th pct" if ind == "fwixd" else "fire-season days"})')
            ax.grid(alpha=.3, axis='y')
    handles = [plt.Rectangle((0, 0), 1, 1, color=GCOL[g]) for g in GWL_ORDER]
    axs[0, 0].legend(handles, [GWL_LABEL[g] for g in GWL_ORDER], fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG}/03_persistence.png', dpi=150); plt.show()
    persist = pd.DataFrame(prow).round(3); persist.to_csv(f'{RES}/03_persistence.csv', index=False)
    display(persist.pivot_table(index=['region', 'indicator', 'model'], columns='gwl', values='likelihood_ratio')[GWL_ORDER].round(1))

## 8. Maps: chance per year of beating the local ERA5 record (2.5° cells, multi-model mean) → GeoTIFF

In [ ]:
import rioxarray  # noqa
obs_rec = obs_g[f'fwixx_{OBS_VERSION}'].max('year')
o_ref = obs_g[f'fwixx_{OBS_VERSION}'].sel(year=slice(*REF))
land = obs_rec.notnull()

def cell_pool(m, g):
    da = LE['fwixx'].where(LE['fwixx'].esm == m, drop=True)
    w = win[(win.esm == m) & (win.gwl == g)].set_index('run')
    parts = [da.sel(run=r, year=slice(w.loc[r, 'start'], w.loc[r, 'end'])).drop_vars(
             [c for c in da.coords if c not in ('year', 'lat', 'lon')]) for r in da.run.values if r in w.index]
    return xr.concat(parts, 'year')

PMAP = {}
for g in GWL_ORDER:
    ps = []
    for m in MODELS:
        da = LE['fwixx'].where(LE['fwixx'].esm == m, drop=True)
        mr = da.sel(year=slice(*REF)).stack(s=('run', 'year'))
        pool = cell_pool(m, g)
        bc = (pool - mr.mean('s')) * (o_ref.std('year', ddof=1) / mr.std('s', ddof=1)) + o_ref.mean('year')
        ps.append((bc > obs_rec).mean('year'))
    PMAP[g] = (100 * xr.concat(ps, 'model').mean('model')).where(land)

def to_tif(da, path):
    da = da.rename(lat='y', lon='x').sortby('y', ascending=False).astype('float32')
    da.rio.write_crs('EPSG:4326', inplace=True).rio.to_raster(path)

fig, axs = plt.subplots(1, len(GWL_ORDER), figsize=(4.4 * len(GWL_ORDER), 4))
for ax, g in zip(axs, GWL_ORDER):
    PMAP[g].plot(ax=ax, cmap='magma_r', vmin=0, vmax=max(5, float(PMAP['3.0'].quantile(.95))),
                 cbar_kwargs=dict(label='% chance per year'))
    if regions_gdf is not None:
        regions_gdf.boundary.plot(ax=ax, color='c', lw=1, aspect=None)
    ax.set_title(f'P(beat ERA5 record) {GWL_LABEL[g]}')
    to_tif(PMAP[g], f'{MAPS}/p_record_fwixx_{g.replace(".", "p")}.tif')
plt.tight_layout(); plt.savefig(f'{FIG}/03_maps_record_probability.png', dpi=150); plt.show()
to_tif(obs_rec.where(land), f'{MAPS}/era5_record_fwixx_g025.tif')
print('GeoTIFFs written to', MAPS)

## 9. Why UNSEEN? A GEV fit to the observed record alone vs the large ensemble (today)

In [ ]:
gev_rows = []
for reg in REGIONS:
    o = OBS[(reg, 'fwixx')]; rec = o.max()
    recent = o.loc[1991:].values
    c, loc, scale = stats.genextreme.fit(recent)
    p_gev = stats.genextreme.sf(rec, c, loc, scale)
    boots = []
    for _ in range(500):
        smp = rng.choice(recent, recent.size, replace=True)
        try:
            cb, lb, sb = stats.genextreme.fit(smp); boots.append(stats.genextreme.sf(rec, cb, lb, sb))
        except Exception:
            pass
    lo, hi = np.percentile(boots, [5, 95])
    un = results[(results.region == reg) & (results.indicator == 'fwixx') & (results.gwl == 'today') &
                 results.model.str.startswith('MULTI')].iloc[0]
    gev_rows.append(dict(region=reg, record=rec, gev_obs_return_period=1 / p_gev if p_gev > 0 else np.inf,
                         gev_RP_90CI=f'{1/hi if hi > 0 else np.inf:.0f}–{1/lo if lo > 0 else np.inf:.0f}',
                         unseen_today_return_period=un.return_period_yr,
                         unseen_range_across_models=f'{1/un.p_hi90 if un.p_hi90 > 0 else np.inf:.0f}–{1/un.p_lo90 if un.p_lo90 > 0 else np.inf:.0f}'))
gev = pd.DataFrame(gev_rows); gev.to_csv(f'{RES}/03_gev_vs_unseen.csv', index=False)
gev

## 10. Key numbers for the write-up

In [ ]:
lines = []
for reg in REGIONS:
    r = mm[(mm.region == reg) & (mm.indicator == 'fwixx')].set_index('gwl')
    t = r.loc['today']
    lines.append(f"**{reg}** — ERA5 record annual-max FWI {t.record:.1f} ({int(t.record_year)}). Today (≈1.3 °C) the "
                 f"large ensembles give a {100*t.p_annual:.1f}% chance per year of beating it (models: {100*t.p_lo90:.1f}–"
                 f"{100*t.p_hi90:.1f}%), i.e. {100*t.p_in_10yr:.0f}% chance in the next 10 years; the worst plausible "
                 f"value is {t.worst:.1f} ({t.worst_pct_above_record:+.0f}% vs record). "
                 + ', '.join(f"{GWL_LABEL[g]}: {100*r.loc[g].p_annual:.1f}%/yr" for g in GWL_ORDER[1:]) + '.')
open(f'{RES}/03_key_numbers.md', 'w').write('\n\n'.join(lines))
print('\n\n'.join(lines))

## 11. Final maps in QGIS (≈30 min)
Download `MyDrive/FWI_India/maps/` to your computer, then in QGIS:
1. **Layer ▸ Add Layer ▸ Add Raster Layer** → `p_record_fwixx_today.tif`, `…_1p5.tif`, `…_2p0.tif`, `…_3p0.tif`, and `era5_fwixx_record_025.tif`.
2. **Layer ▸ Add Vector Layer** → `regions.geojson` (no fill, black 0.6 mm outline). Add an India/state boundary layer
   (e.g. GADM level 1 or the Survey of India boundary) and Natural Earth coastlines for context.
3. Raster styling: **Properties ▸ Symbology ▸ Singleband pseudocolor**, colour ramp *Magma* (inverted), Min 0 / Max = the 3 °C
   layer's 95th percentile (use the same min/max for all four warming levels so they are comparable). Set **Resampling ▸ Nearest**
   so the 2.5° cells stay crisp. Right-click a styled layer ▸ *Styles ▸ Copy Style* → paste on the others.
4. Project CRS: EPSG:4326 is fine; for a publication-style map use *EPSG:7755 (WGS 84 / India NSF LCC)*.
5. **Project ▸ New Print Layout**: four map items (one per warming level, locked layers), one shared legend
   ("% chance per year of beating the 1951–2026 ERA5 record"), scale bar, north arrow, data credits
   (ERA5-Land/Copernicus; CMIP6 FWI, Quilcaille et al. 2023). Export at 300 dpi PNG/PDF.
6. Optional second layout: `era5_fwixx_record_025.tif` + `era5_fwixx_record_year_025.tif` (categorised by year) with
   MODIS burned area of the record season (exported from GEE) as an overlay.